# Model Diagnostics by Province and Season

This notebook evaluates the already-trained models on a chronological holdout split and reports metrics globally, by province, and by season.

## Step 1: Import Libraries
This cell imports required packages and sets up the project path so notebook code can import modules from src.

In [2]:
import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.preprocess import add_time_features, clean_weather_data, load_dataset

## Step 2: Set Paths and Load Metadata
This cell defines key settings, checks required files, and loads model comparison metadata.

In [3]:
TARGET_COL = 'temp_max'
TEST_FRACTION = 0.20

dataset_path = Path('../data/cambodia_weather.csv')
compare_path = Path('../artifacts/preprocessors/model_comparison_metadata.json')

if not dataset_path.exists():
    raise FileNotFoundError(f'Missing dataset: {dataset_path}')
if not compare_path.exists():
    raise FileNotFoundError(
        'Missing comparison metadata. Run: python -m src.models.train_compare --target temp_max'
    )

with compare_path.open('r', encoding='utf-8') as f:
    compare_meta = json.load(f)

feature_columns = compare_meta['feature_columns']
model_entries = compare_meta['models']

## Step 3: Prepare Data and Create Time Split
This cell cleans data, builds features, and creates a chronological train/test split to avoid time leakage.

In [5]:
df = load_dataset(dataset_path)
df = add_time_features(df)
df = clean_weather_data(df)

required_cols = ['province', TARGET_COL, 'date']
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f'Missing required columns in dataset: {missing_cols}')

numeric_cols = [
    'temp_min', 'rain', 'wind_speed', 'lat', 'lon',
    'year', 'month', 'day', 'dayofweek',
    'month_sin', 'month_cos', 'dayofweek_sin', 'dayofweek_cos'
]
for col in numeric_cols + [TARGET_COL]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

base = df[numeric_cols + ['province', TARGET_COL, 'date']].dropna().copy()
province_dummies = pd.get_dummies(base['province'].astype(str), prefix='province').astype(float)
X_all = pd.concat([base[numeric_cols].astype(float), province_dummies], axis=1)
X_all = X_all.reindex(columns=feature_columns, fill_value=0.0).astype(float)
y_all = base[TARGET_COL].astype(float)
context_all = base[['date', 'province', 'month']].copy()
context_all['date'] = pd.to_datetime(context_all['date'], errors='coerce')

valid_idx = context_all['date'].notna()
X_all = X_all.loc[valid_idx]
y_all = y_all.loc[valid_idx]
context_all = context_all.loc[valid_idx]

order = context_all['date'].sort_values().index
X_all = X_all.loc[order].reset_index(drop=True)
y_all = y_all.loc[order].reset_index(drop=True)
context_all = context_all.loc[order].reset_index(drop=True)

split_idx = int(len(X_all) * (1.0 - TEST_FRACTION))
X_train, X_test = X_all.iloc[:split_idx], X_all.iloc[split_idx:]
y_train, y_test = y_all.iloc[:split_idx], y_all.iloc[split_idx:]
ctx_test = context_all.iloc[split_idx:].reset_index(drop=True)

print(f'Train rows: {len(X_train):,}')
print(f'Test rows: {len(X_test):,}')

Train rows: 16,456
Test rows: 4,114


## Step 4: Define Helper Functions
This cell adds reusable functions for regression metrics and wet/dry season labeling.

In [6]:
def regression_metrics(y_true, y_pred):
    return {
        'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'r2': float(r2_score(y_true, y_pred)),
    }

def season_from_month(month_num):
    if pd.isna(month_num):
        return 'Unknown'
    month_num = int(month_num)
    return 'Wet (May-Oct)' if month_num in {5, 6, 7, 8, 9, 10} else 'Dry (Nov-Apr)'

## Step 5: Evaluate Overall Model Performance
This cell loads each trained model, predicts on the test set, and shows global RMSE, MAE, and R2.

In [7]:
global_rows = []
predictions = {}

for model_key, model_info in model_entries.items():
    model_path = Path(model_info['model_path'])
    if not model_path.exists():
        raise FileNotFoundError(f'Model file not found for {model_key}: {model_path}')

    model = joblib.load(model_path)
    y_pred = model.predict(X_test)
    predictions[model_key] = y_pred

    m = regression_metrics(y_test, y_pred)
    global_rows.append({
        'model': model_key,
        'model_type': model_info.get('model_type', 'N/A'),
        'rmse': m['rmse'],
        'mae': m['mae'],
        'r2': m['r2'],
        'train_rows': len(X_train),
        'test_rows': len(X_test),
    })

global_metrics = pd.DataFrame(global_rows).sort_values('rmse').reset_index(drop=True)
display(global_metrics.style.format({'rmse': '{:.3f}', 'mae': '{:.3f}', 'r2': '{:.3f}'}))

,model,model_type,rmse,mae,r2,train_rows,test_rows
0,random_forest,RandomForestRegressor,0.688,0.484,0.948,16456,4114
1,decision_tree,DecisionTreeRegressor,0.861,0.502,0.919,16456,4114
2,linear_regression,LinearRegression,1.478,1.136,0.761,16456,4114


## Step 6: Evaluate by Province
This cell calculates model performance for each province to reveal location-specific strengths and weaknesses.

In [8]:
province_rows = []

for model_key, y_pred in predictions.items():
    tmp = pd.DataFrame({
        'province': ctx_test['province'].astype(str).values,
        'y_true': y_test.values,
        'y_pred': y_pred,
    })

    for province, grp in tmp.groupby('province'):
        m = regression_metrics(grp['y_true'], grp['y_pred'])
        province_rows.append({
            'model': model_key,
            'province': province,
            'rows': len(grp),
            'rmse': m['rmse'],
            'mae': m['mae'],
            'r2': m['r2'],
        })

province_metrics = pd.DataFrame(province_rows).sort_values(['model', 'rmse']).reset_index(drop=True)
province_metrics.head(20)

,model,province,rows,rmse,mae,r2
0,decision_tree,Sihanoukville,823,0.615550,0.332333,0.839942
1,decision_tree,Mondulkiri,823,0.772625,0.357428,0.926368
2,decision_tree,Siem Reap,823,0.866711,0.566210,0.849555
3,decision_tree,Phnom Penh,822,0.955838,0.635697,0.790998
4,decision_tree,Battambang,823,1.031108,0.616276,0.853018
5,linear_regression,Sihanoukville,823,1.190160,0.944171,0.401642
6,linear_regression,Siem Reap,823,1.235795,0.984034,0.694142
7,linear_regression,Phnom Penh,822,1.262242,0.981023,0.635526
8,linear_regression,Battambang,823,1.620491,1.261639,0.636963
9,linear_regression,Mondulkiri,823,1.940109,1.510507,0.535718


## Step 7: Evaluate by Season
This cell compares model performance between wet and dry seasons using grouped metrics.

In [9]:
season_rows = []

for model_key, y_pred in predictions.items():
    tmp = pd.DataFrame({
        'season': ctx_test['month'].map(season_from_month).values,
        'y_true': y_test.values,
        'y_pred': y_pred,
    })

    for season, grp in tmp.groupby('season'):
        m = regression_metrics(grp['y_true'], grp['y_pred'])
        season_rows.append({
            'model': model_key,
            'season': season,
            'rows': len(grp),
            'rmse': m['rmse'],
            'mae': m['mae'],
            'r2': m['r2'],
        })

season_metrics = pd.DataFrame(season_rows).sort_values(['season', 'rmse']).reset_index(drop=True)
season_metrics

,model,season,rows,rmse,mae,r2
0,random_forest,Dry (Nov-Apr),2274,0.644788,0.461918,0.956590
1,decision_tree,Dry (Nov-Apr),2274,0.832576,0.494095,0.927623
2,linear_regression,Dry (Nov-Apr),2274,1.519147,1.166001,0.759034
3,random_forest,Wet (May-Oct),1840,0.738415,0.510513,0.933503
4,decision_tree,Wet (May-Oct),1840,0.894141,0.510778,0.902498
5,linear_regression,Wet (May-Oct),1840,1.426329,1.099622,0.751892


## Optional: Save Diagnostic Tables
Run the next cell to export diagnostics to CSV files in artifacts/preprocessors.

## Step 8: Export Diagnostic Tables
This cell saves global, province, and season metric tables as CSV files for reporting and reuse.

In [10]:
output_dir = Path('../artifacts/preprocessors')
output_dir.mkdir(parents=True, exist_ok=True)

global_metrics.to_csv(output_dir / 'diagnostics_global_metrics.csv', index=False)
province_metrics.to_csv(output_dir / 'diagnostics_province_metrics.csv', index=False)
season_metrics.to_csv(output_dir / 'diagnostics_season_metrics.csv', index=False)

print('Saved diagnostic CSV files to artifacts/preprocessors.')

Saved diagnostic CSV files to artifacts/preprocessors.
